# BBQ Bias Evaluation â€” First Working Version

This MVP evaluates BBQ Race_ethnicity items with one local Hugging Face model. It reports core accuracy, unknown-answer, invalid-answer, and bias metrics. Run top to bottom; start with the smoke sample.

## 1. Setup

Load the project configuration and optional Hugging Face token from `.env`.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "biaseval").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
from biaseval.config import load_config

load_dotenv(ROOT / ".env")
config = load_config(ROOT / "configs" / "default.yaml")
SEED = int(config["seed"])
print(f"Repository: {ROOT}; seed: {SEED}")

## 2. Choose the model and run size

Edit these values to change the Hugging Face model, scoring method, or prompt condition. `RUN_FULL=False` is the recommended first check.

In [ ]:
MODEL_ID = config["model_id"]  # e.g. Qwen/Qwen3-1.7B or a local model directory
DEVICE = config["device"]      # auto, cpu, cuda, or mps
SCORING_MODE = config["scoring_mode"]  # auto, loglik, or generation
PROMPT_CONDITION = config["prompt_condition"]  # plain or debias
SYSTEM_PROMPT = config["prompts"][PROMPT_CONDITION]["system"]
RUN_FULL = False
MAX_NEW_TOKENS = int(config["max_new_tokens"])
CATEGORY = "Race_ethnicity"
if PROMPT_CONDITION not in {"plain", "debias"}:
    raise ValueError("PROMPT_CONDITION must be plain or debias")
print({"model": MODEL_ID, "device": DEVICE, "scoring": SCORING_MODE, "prompt": PROMPT_CONDITION})

## 3. Load BBQ data

The loader downloads only the selected category and joins its official `target_loc` metadata.

In [ ]:
import pandas as pd
from biaseval.data.bbq import load_bbq
from biaseval.data.sampling import sample_bbq

bbq = load_bbq(categories=[CATEGORY], raw_dir=ROOT / "data" / "raw" / "bbq")
display(bbq.groupby(["condition", "polarity"], observed=True).size().rename("items").to_frame())
print(f"{CATEGORY}: {len(bbq):,} usable items (expected 6,880)")

## 4. Prepare the smoke sample and load the local model

The sample is reproducible and balanced across ambiguous/disambiguated and negative/non-negative items. Model weights load only when this cell is run.

In [ ]:
from biaseval.backends.hf_local import HuggingFaceBackend

evaluation_data = (
    bbq.copy() if RUN_FULL
    else sample_bbq(bbq, int(config["smoke_items"]), seed=SEED)
)
backend = HuggingFaceBackend(
    MODEL_ID,
    device=DEVICE,
    token=os.getenv("HF_TOKEN") or None,
)
print("Backend:", backend.metadata)
print(f"Items queued: {len(evaluation_data)}")

## 5. Inspect one example

Check the prompt and the answer scoring before running the full selected sample.

In [ ]:
from biaseval.prompts import build_prompt
from biaseval.parsing import parse_answer

item = evaluation_data.iloc[0].to_dict()
prompt = build_prompt(item, system_prompt=SYSTEM_PROMPT, tokenizer=backend.tokenizer)
print(prompt)
if SCORING_MODE == "generation":
    raw = backend.generate([prompt], max_new_tokens=MAX_NEW_TOKENS)[0]
    print("Raw answer:", raw)
    print("Parsed answer:", parse_answer(raw, item["options"]))
else:
    scores = backend.score_options([prompt], len(item["options"]))[0]
    print("Option scores:", scores)
    print("Predicted option:", int(max(range(len(scores)), key=scores.__getitem__)))

## 6. Run and review the evaluation

Predictions, resolved run configuration, and metrics are saved under `results/runs/`. Set `RUN_FULL=True` only after the smoke sample works.

In [ ]:
from biaseval.runner import run_evaluation

predictions, metrics, run_dir = run_evaluation(
    evaluation_data,
    backend,
    results_dir=ROOT / "results",
    model_id=MODEL_ID,
    prompt_condition=PROMPT_CONDITION,
    system_prompt=SYSTEM_PROMPT,
    scoring_mode=SCORING_MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)
display(pd.DataFrame.from_dict(metrics["overall"], orient="index", columns=["value"]))
display(pd.DataFrame.from_dict(
    {key: value for key, value in metrics.items() if key.startswith("category_condition|")},
    orient="index",
))
print("Saved run:", run_dir)